# Discovery y Perfilado de Datos — Sistema CRM

In [1]:
import os
import pandas as pd
from sqlalchemy import create_engine

In [2]:
user = os.environ.get("WAREHOUSE_DB_USER", "postgres")
password = os.environ.get("WAREHOUSE_DB_PASSWORD", "postgres")
db = os.environ.get("WAREHOUSE_DB_NAME", "warehouse")
host = os.environ.get("WAREHOUSE_DB_HOST", "warehouse-db")
port = os.environ.get("WAREHOUSE_DB_PORT", "5432")

In [3]:
engine = create_engine(f"postgresql+psycopg2://{user}:{password}@{host}:{port}/{db}")

### Conexión crm y billing (contacts.email y customers.email)

In [4]:
#Cuántos contactos de CRM son también clientes de billing
query = """
SELECT
  COUNT(*) AS total_contacts,
  COUNT(c.customer_id) AS con_match_en_billing,
  COUNT(*) - COUNT(c.customer_id) AS sin_match
FROM bronze.crm_contacts ct
LEFT JOIN bronze.billing_customers c ON c.email = ct.email;
"""
df = pd.read_sql(query, engine)
df

,total_contacts,con_match_en_billing,sin_match
0,15000,1,14999


In [5]:
#Detalle de los que sí matchean (para inspección)
query = """
SELECT ct.contact_id, ct.email, ct.account_id, c.customer_id, c.segment
FROM bronze.crm_contacts ct
JOIN bronze.billing_customers c ON c.email = ct.email;
"""
df = pd.read_sql(query, engine)
df

,contact_id,email,account_id,customer_id,segment
0,CON-0010616,paula.soto6861@demo.io,ACC-0004787,CUS-0003142,retail


### Tabla accounts

In [6]:
#Perfil general
query = """
SELECT
  COUNT(*) AS total_rows,
  COUNT(DISTINCT account_id) AS ids,
  COUNT(*) FILTER (WHERE industry IS NULL) AS industry,
  COUNT(*) FILTER (WHERE country IS NULL) AS country,
  COUNT(*) FILTER (WHERE annual_revenue IS NULL) AS annual_revenue,
  COUNT(*) FILTER (WHERE employees IS NULL) AS employees,
  COUNT(*) FILTER (WHERE created_at IS NULL) AS created_at
FROM bronze.crm_accounts;
"""
df = pd.read_sql(query, engine)
df

,total_rows,ids,industry,country,annual_revenue,employees,created_at
0,5000,5000,0,0,0,0,0


In [7]:
#Duplicados de name (posibles cuentas duplicadas)
query = """
SELECT name, COUNT(*) AS n
FROM bronze.crm_accounts
GROUP BY name
HAVING COUNT(*) > 1;
"""
df = pd.read_sql(query, engine)
df

,name,n
0,Patagonia Bank,5
1,Salmon Labs,10
2,Maipo Capital,4
3,Azteca Mining,13
4,Norte Cloud,7
...,...,...
594,Azteca Bank,11
595,Azteca Data,17
596,Atacama Mining,11
597,Pacific Retail,6


In [8]:
#Cardinalidad de industry y country
query = """
SELECT industry, COUNT(*) AS n
FROM bronze.crm_accounts
GROUP BY industry
ORDER BY n DESC;
"""
df = pd.read_sql(query, engine)
df

,industry,n
0,tech,909
1,finance,751
2,retail,739
3,health,604
4,manufacturing,579
5,education,527
6,services,495
7,energy,396


In [9]:
query = """
SELECT country, COUNT(*) AS n
FROM bronze.crm_accounts
GROUP BY country
ORDER BY n DESC;
"""
df = pd.read_sql(query, engine)
df

,country,n
0,CL,1499
1,MX,618
2,AR,601
3,PE,512
4,CO,487
5,BR,468
6,ES,428
7,US,387


In [10]:
#Distribución de annual_revenue y employees (con validación de negativos)
query = """
SELECT
  MIN(annual_revenue::numeric) AS ingreso_min,
  MAX(annual_revenue::numeric) AS ingreso_max,
  AVG(annual_revenue::numeric) AS ingreso_promedio
FROM bronze.crm_accounts;
"""
df = pd.read_sql(query, engine)
df

,ingreso_min,ingreso_max,ingreso_promedio
0,11003.91,64944915.65,924675.311754


In [11]:
query = """
SELECT MIN(employees::numeric) AS empleados_min, MAX(employees::numeric) AS empleados_max
FROM bronze.crm_accounts;
"""
df = pd.read_sql(query, engine)
df

,empleados_min,empleados_max
0,1.0,6521.0


In [12]:
query = """
SELECT *
FROM bronze.crm_accounts
WHERE annual_revenue::numeric < 0 OR employees::numeric < 0;
"""
df = pd.read_sql(query, engine)
df

,account_id,name,industry,country,annual_revenue,employees,created_at


In [13]:
#Validar formato y rango de created_at
query = """
SELECT created_at, COUNT(*) AS n
FROM bronze.crm_accounts
WHERE created_at IS NOT NULL AND created_at !~ '^\d{4}-\d{2}-\d{2}'
GROUP BY created_at;
"""
df = pd.read_sql(query, engine)
df

,created_at,n


In [14]:
query = """
SELECT MIN(created_at::date) AS cuenta_mas_antigua, MAX(created_at::date) AS cuenta_mas_reciente
FROM bronze.crm_accounts;
"""
df = pd.read_sql(query, engine)
df

,cuenta_mas_antigua,cuenta_mas_reciente
0,2018-01-01,2025-12-30


### Tabla contacts

In [15]:
#Perfil general
query = """
SELECT
  COUNT(*) AS total_rows,
  COUNT(DISTINCT contact_id) AS ids,
  COUNT(*) FILTER (WHERE phone IS NULL) AS phone,
  COUNT(*) FILTER (WHERE title IS NULL) AS title,
  COUNT(*) FILTER (WHERE email IS NULL) AS email,
  COUNT(*) FILTER (WHERE created_at IS NULL) AS created_at,
  COUNT(*) FILTER (WHERE account_id IS NULL) AS account_id
FROM bronze.crm_contacts;
"""
df = pd.read_sql(query, engine)
df

,total_rows,ids,phone,title,email,created_at,account_id
0,15000,15000,0,0,0,0,0


In [16]:
#Duplicados de email
query = """
SELECT email, COUNT(*) AS n
FROM bronze.crm_contacts
GROUP BY email
HAVING COUNT(*) > 1;
"""
df = pd.read_sql(query, engine)
df

,email,n
0,macarena.ortiz4996@lake.local,2
1,ignacio.tapia5946@mail.test,2


In [17]:
#Cardinalidad de title
query = """
SELECT title, COUNT(*) AS n
FROM bronze.crm_contacts
GROUP BY title
ORDER BY n DESC;
"""
df = pd.read_sql(query, engine)
df

,title,n
0,Analyst,1537
1,Engineer,1529
2,CFO,1519
3,Manager,1514
4,VP,1502
5,Specialist,1500
6,CEO,1491
7,CTO,1487
8,Sales Rep,1469
9,Director,1452


In [18]:
#account_id huérfano (contacto "fantasma")
query = """
SELECT ct.contact_id, ct.account_id
FROM bronze.crm_contacts ct
LEFT JOIN bronze.crm_accounts a ON a.account_id = ct.account_id
WHERE ct.account_id IS NOT NULL AND a.account_id IS NULL;
"""
df = pd.read_sql(query, engine)
df

,contact_id,account_id


In [19]:
#Contacto creado antes que su cuenta (imposible cronológicamente)
query = """
SELECT ct.contact_id, ct.created_at AS contacto_creado,
       a.account_id, a.created_at AS cuenta_creada
FROM bronze.crm_contacts ct
JOIN bronze.crm_accounts a ON a.account_id = ct.account_id
WHERE ct.created_at::date < a.created_at::date;
"""
df = pd.read_sql(query, engine)
df

,contact_id,contacto_creado,account_id,cuenta_creada
0,CON-0000001,2019-11-14 15:32:18,ACC-0003994,2025-10-30 01:51:28
1,CON-0000005,2019-10-04 09:46:14,ACC-0001288,2024-07-01 17:05:04
2,CON-0000012,2020-07-09 21:05:28,ACC-0002639,2024-04-05 03:01:06
3,CON-0000013,2023-08-08 23:08:02,ACC-0004002,2025-09-05 11:54:55
4,CON-0000015,2021-02-26 04:54:00,ACC-0001051,2024-11-09 01:35:33
...,...,...,...,...
7446,CON-0014992,2018-06-22 23:33:51,ACC-0002911,2024-08-24 01:02:42
7447,CON-0014993,2018-11-16 02:32:31,ACC-0000372,2021-11-09 06:01:12
7448,CON-0014994,2018-05-28 21:32:36,ACC-0000587,2019-05-19 10:29:19
7449,CON-0014997,2018-05-04 01:09:22,ACC-0000944,2022-07-02 12:22:24


### Tabla leads

In [20]:
#Perfil general
query = """
SELECT
  COUNT(*) AS total_rows,
  COUNT(DISTINCT lead_id) AS ids,
  COUNT(*) FILTER (WHERE source IS NULL) AS source,
  COUNT(*) FILTER (WHERE status IS NULL) AS status,
  COUNT(*) FILTER (WHERE score IS NULL) AS score,
  COUNT(*) FILTER (WHERE created_at IS NULL) AS created_at
FROM bronze.crm_leads;
"""
df = pd.read_sql(query, engine)
df

,total_rows,ids,source,status,score,created_at
0,2000,2000,0,0,0,0


In [21]:
#Cardinalidad de source y status
query = """
SELECT source, COUNT(*) AS n
FROM bronze.crm_leads
GROUP BY source
ORDER BY n DESC;
"""
df = pd.read_sql(query, engine)
df

,source,n
0,web,810
1,referral,402
2,event,302
3,ads,282
4,cold_call,204


In [22]:
query = """
SELECT status, COUNT(*) AS n
FROM bronze.crm_leads
GROUP BY status
ORDER BY n DESC;
"""
df = pd.read_sql(query, engine)
df

,status,n
0,new,594
1,contacted,525
2,qualified,395
3,lost,281
4,converted,205


In [23]:
#Distribución de score
query = """
SELECT MIN(score::numeric) AS score_min, MAX(score::numeric) AS score_max, AVG(score::numeric) AS score_promedio
FROM bronze.crm_leads;
"""
df = pd.read_sql(query, engine)
df

,score_min,score_max,score_promedio
0,0.0,100.0,50.0965


In [24]:
#Duplicados de email
query = """
SELECT email, COUNT(*) AS n
FROM bronze.crm_leads
GROUP BY email
HAVING COUNT(*) > 1;
"""
df = pd.read_sql(query, engine)
df

,email,n


In [25]:
#Leads que ya existen como contacto (conversión no modelada en el esquema)
query = """
SELECT
  COUNT(*) AS total_leads,
  COUNT(ct.contact_id) AS con_match_en_contacts,
  COUNT(*) - COUNT(ct.contact_id) AS sin_match
FROM bronze.crm_leads l
LEFT JOIN bronze.crm_contacts ct ON ct.email = l.email;
"""
df = pd.read_sql(query, engine)
df

,total_leads,con_match_en_contacts,sin_match
0,2000,0,2000


### Tabla opportunities

In [26]:
#Perfil general
query = """
SELECT
  COUNT(*) AS total_rows,
  COUNT(DISTINCT opportunity_id) AS ids,
  COUNT(*) FILTER (WHERE stage IS NULL) AS stage,
  COUNT(*) FILTER (WHERE amount IS NULL) AS amount,
  COUNT(*) FILTER (WHERE close_date IS NULL) AS close_date,
  COUNT(*) FILTER (WHERE created_at IS NULL) AS created_at,
  COUNT(*) FILTER (WHERE account_id IS NULL) AS account_id
FROM bronze.crm_opportunities;
"""
df = pd.read_sql(query, engine)
df

,total_rows,ids,stage,amount,close_date,created_at,account_id
0,3000,3000,0,0,0,0,0


In [27]:
#Cardinalidad de stage
query = """
SELECT stage, COUNT(*) AS n
FROM bronze.crm_opportunities
GROUP BY stage
ORDER BY n DESC;
"""
df = pd.read_sql(query, engine)
df

,stage,n
0,prospect,621
1,qualification,611
2,proposal,569
3,won,476
4,negotiation,420
5,lost,303


In [28]:
#Distribución de amount
query = """
SELECT MIN(amount::numeric) AS monto_min, MAX(amount::numeric) AS monto_max, AVG(amount::numeric) AS monto_promedio
FROM bronze.crm_opportunities;
"""
df = pd.read_sql(query, engine)
df

,monto_min,monto_max,monto_promedio
0,699.92,687007.8,37921.932363


In [29]:
query = """
SELECT stage,
       MIN(amount::numeric) AS monto_min, MAX(amount::numeric) AS monto_max, AVG(amount::numeric) AS monto_promedio,
       COUNT(*) AS n
FROM bronze.crm_opportunities
GROUP BY stage
ORDER BY stage;
"""
df = pd.read_sql(query, engine)
df

,stage,monto_min,monto_max,monto_promedio,n
0,lost,1149.03,687007.80,39408.812211,303
1,negotiation,1675.63,380322.19,36597.337619,420
2,proposal,699.92,486753.03,39056.054569,569
3,prospect,1914.32,407801.51,36762.757085,621
4,qualification,790.56,541485.25,37710.256727,611
5,won,1418.94,508548.11,38572.502374,476


In [30]:
#close_date anterior a created_at (imposible cronológicamente)
query = """
SELECT opportunity_id, created_at, close_date
FROM bronze.crm_opportunities
WHERE close_date IS NOT NULL
  AND close_date::date < created_at::date;
"""
df = pd.read_sql(query, engine)
df

,opportunity_id,created_at,close_date
0,OPP-0000002,2025-05-25 03:05:15,2023-09-28
1,OPP-0000005,2024-11-30 06:25:22,2023-08-30
2,OPP-0000009,2024-07-08 09:42:27,2023-08-29
3,OPP-0000012,2025-03-09 20:57:04,2023-03-03
4,OPP-0000013,2023-10-06 12:46:38,2023-06-03
...,...,...,...
1024,OPP-0002991,2025-03-17 05:26:03,2024-08-21
1025,OPP-0002993,2024-07-05 01:49:52,2023-01-26
1026,OPP-0002995,2025-05-23 12:23:37,2023-11-14
1027,OPP-0002996,2024-08-14 03:33:19,2023-04-23


In [31]:
#Oportunidades en stage terminal sin close_date
query = """
-- Ajustar los valores de stage 'closed_won'/'closed_lost' a los reales que arroje 4.2
SELECT *
FROM bronze.crm_opportunities
WHERE stage IN ('closed_won', 'closed_lost')
  AND close_date IS NULL;
"""
df = pd.read_sql(query, engine)
df

,opportunity_id,name,stage,amount,close_date,created_at,account_id


In [32]:
#account_id huérfano
query = """
SELECT o.opportunity_id, o.account_id
FROM bronze.crm_opportunities o
LEFT JOIN bronze.crm_accounts a ON a.account_id = o.account_id
WHERE o.account_id IS NOT NULL AND a.account_id IS NULL;
"""
df = pd.read_sql(query, engine)
df

,opportunity_id,account_id


### Tabla opportunity_contacts

In [33]:
#Perfil general (tabla puente, sin PK propia)
query = """
SELECT
  COUNT(*) AS total_rows,
  COUNT(DISTINCT opportunity_id) AS opportunities_distintas,
  COUNT(DISTINCT contact_id) AS contacts_distintos,
  COUNT(*) FILTER (WHERE role IS NULL) AS role
FROM bronze.crm_opportunity_contacts;
"""
df = pd.read_sql(query, engine)
df

,total_rows,opportunities_distintas,contacts_distintos,role
0,6000,2586,4955,0


In [34]:
#Duplicados de la combinación (opportunity_id, contact_id)
query = """
SELECT opportunity_id, contact_id, COUNT(*) AS n
FROM bronze.crm_opportunity_contacts
GROUP BY opportunity_id, contact_id
HAVING COUNT(*) > 1;
"""
df = pd.read_sql(query, engine)
df

,opportunity_id,contact_id,n


In [35]:
#Cardinalidad de role
query = """
SELECT role, COUNT(*) AS n
FROM bronze.crm_opportunity_contacts
GROUP BY role
ORDER BY n DESC;
"""
df = pd.read_sql(query, engine)
df

,role,n
0,influencer,1253
1,end_user,1232
2,financial,1200
3,technical,1159
4,decision_maker,1156


In [36]:
#Integridad referencial (ambos lados)
query = """
SELECT oc.opportunity_id, oc.contact_id
FROM bronze.crm_opportunity_contacts oc
LEFT JOIN bronze.crm_opportunities o ON o.opportunity_id = oc.opportunity_id
WHERE o.opportunity_id IS NULL;
"""
df = pd.read_sql(query, engine)
df

,opportunity_id,contact_id


In [37]:
query = """
SELECT oc.opportunity_id, oc.contact_id
FROM bronze.crm_opportunity_contacts oc
LEFT JOIN bronze.crm_contacts ct ON ct.contact_id = oc.contact_id
WHERE ct.contact_id IS NULL;
"""
df = pd.read_sql(query, engine)
df

,opportunity_id,contact_id


In [38]:
#Cruce role x title (para revisar inconsistencias a simple vista)
query = """
SELECT oc.role, ct.title, COUNT(*) AS n
FROM bronze.crm_opportunity_contacts oc
JOIN bronze.crm_contacts ct ON ct.contact_id = oc.contact_id
GROUP BY oc.role, ct.title
ORDER BY oc.role, n DESC;
"""
df = pd.read_sql(query, engine)
df

,role,title,n
0,decision_maker,Analyst,134
1,decision_maker,Engineer,125
2,decision_maker,CEO,124
3,decision_maker,VP,121
4,decision_maker,Sales Rep,120
5,decision_maker,CFO,119
6,decision_maker,Manager,114
7,decision_maker,Director,105
8,decision_maker,Specialist,99
9,decision_maker,CTO,95


In [39]:
#Integridad referencial (ambos lados)
query = """
SELECT oc.opportunity_id, oc.contact_id
FROM bronze.crm_opportunity_contacts oc
LEFT JOIN bronze.crm_opportunities o ON o.opportunity_id = oc.opportunity_id
WHERE o.opportunity_id IS NULL;
"""
df = pd.read_sql(query, engine)
df

,opportunity_id,contact_id


In [40]:
query = """
SELECT oc.opportunity_id, oc.contact_id
FROM bronze.crm_opportunity_contacts oc
LEFT JOIN bronze.crm_contacts ct ON ct.contact_id = oc.contact_id
WHERE ct.contact_id IS NULL;
"""
df = pd.read_sql(query, engine)
df

,opportunity_id,contact_id


In [41]:
#Cruce role x title (para revisar inconsistencias a simple vista)
query = """
SELECT oc.role, ct.title, COUNT(*) AS n
FROM bronze.crm_opportunity_contacts oc
JOIN bronze.crm_contacts ct ON ct.contact_id = oc.contact_id
GROUP BY oc.role, ct.title
ORDER BY oc.role, n DESC;
"""
df = pd.read_sql(query, engine)
df

,role,title,n
0,decision_maker,Analyst,134
1,decision_maker,Engineer,125
2,decision_maker,CEO,124
3,decision_maker,VP,121
4,decision_maker,Sales Rep,120
5,decision_maker,CFO,119
6,decision_maker,Manager,114
7,decision_maker,Director,105
8,decision_maker,Specialist,99
9,decision_maker,CTO,95


### Tabla activities

In [42]:
#Perfil general
query = """
SELECT
  COUNT(*) AS total_rows,
  COUNT(DISTINCT activity_id) AS ids,
  COUNT(*) FILTER (WHERE type IS NULL) AS type,
  COUNT(*) FILTER (WHERE subject IS NULL) AS subject,
  COUNT(*) FILTER (WHERE occurred_at IS NULL) AS occurred_at,
  COUNT(*) FILTER (WHERE contact_id IS NULL) AS contact_id,
  COUNT(*) FILTER (WHERE opportunity_id IS NULL) AS opportunity_id
FROM bronze.crm_activities;
"""
df = pd.read_sql(query, engine)
df

,total_rows,ids,type,subject,occurred_at,contact_id,opportunity_id
0,20000,20000,0,0,0,5976,9985


In [43]:
#Cardinalidad de type
query = """
SELECT type, COUNT(*) AS n
FROM bronze.crm_activities
GROUP BY type
ORDER BY n DESC;
"""
df = pd.read_sql(query, engine)
df

,type,n
0,email,6904
1,call,6093
2,meeting,3016
3,note,2028
4,demo,1959


In [44]:
#¿subject es categoría o es casi-único? (ratio de cardinalidad)
query = """
SELECT
  COUNT(*) AS total_rows,
  COUNT(DISTINCT subject) AS distinct_subjects,
  ROUND(COUNT(DISTINCT subject)::numeric / COUNT(*), 4) AS ratio_unicidad
FROM bronze.crm_activities;
"""
df = pd.read_sql(query, engine)
df

,total_rows,distinct_subjects,ratio_unicidad
0,20000,20000,1.0


In [45]:
#Validar formato y rango de occurred_at
query = """
SELECT occurred_at, COUNT(*) AS n
FROM bronze.crm_activities
WHERE occurred_at IS NOT NULL AND occurred_at !~ '^\d{4}-\d{2}-\d{2}'
GROUP BY occurred_at;
"""
df = pd.read_sql(query, engine)
df

,occurred_at,n


In [46]:
query = """
SELECT MIN(occurred_at::date) AS actividad_mas_antigua, MAX(occurred_at::date) AS actividad_mas_reciente
FROM bronze.crm_activities;
"""
df = pd.read_sql(query, engine)
df

,actividad_mas_antigua,actividad_mas_reciente
0,2023-01-01,2025-12-30


In [47]:
#contact_id y opportunity_id huérfanos
query = """
SELECT a.activity_id, a.contact_id
FROM bronze.crm_activities a
LEFT JOIN bronze.crm_contacts ct ON ct.contact_id = a.contact_id
WHERE a.contact_id IS NOT NULL AND ct.contact_id IS NULL;
"""
df = pd.read_sql(query, engine)
df

,activity_id,contact_id


In [48]:
query = """
SELECT a.activity_id, a.opportunity_id
FROM bronze.crm_activities a
LEFT JOIN bronze.crm_opportunities o ON o.opportunity_id = a.opportunity_id
WHERE a.opportunity_id IS NOT NULL AND o.opportunity_id IS NULL;
"""
df = pd.read_sql(query, engine)
df

,activity_id,opportunity_id


In [49]:
#Actividad ocurrida antes de que existiera el contacto
query = """
SELECT a.activity_id, a.occurred_at, ct.contact_id, ct.created_at AS contacto_creado
FROM bronze.crm_activities a
JOIN bronze.crm_contacts ct ON ct.contact_id = a.contact_id
WHERE a.occurred_at::date < ct.created_at::date;
"""
df = pd.read_sql(query, engine)
df

,activity_id,occurred_at,contact_id,contacto_creado
0,ACT-00000012,2023-11-04 10:52:43,CON-0003354,2024-09-29 22:17:54
1,ACT-00000014,2023-11-26 03:50:33,CON-0006184,2025-05-25 01:03:19
2,ACT-00000017,2023-11-27 15:50:11,CON-0012783,2025-03-29 08:05:52
3,ACT-00000037,2023-01-08 19:07:54,CON-0008262,2024-08-09 20:43:18
4,ACT-00000041,2023-07-02 16:34:04,CON-0006052,2024-06-24 21:44:42
...,...,...,...,...
2574,ACT-00019923,2023-06-14 19:54:58,CON-0009698,2025-09-09 00:48:11
2575,ACT-00019939,2023-11-20 10:25:21,CON-0004828,2024-06-28 06:57:08
2576,ACT-00019984,2023-05-22 09:49:47,CON-0013654,2025-09-10 11:15:10
2577,ACT-00019991,2024-08-17 09:08:48,CON-0003456,2025-08-11 10:41:13


In [50]:
#Actividad ocurrida antes de que existiera la oportunidad
query = """
SELECT a.activity_id, a.occurred_at, o.opportunity_id, o.created_at AS oportunidad_creada
FROM bronze.crm_activities a
JOIN bronze.crm_opportunities o ON o.opportunity_id = a.opportunity_id
WHERE a.occurred_at::date < o.created_at::date;
"""
df = pd.read_sql(query, engine)
df

,activity_id,occurred_at,opportunity_id,oportunidad_creada
0,ACT-00000009,2023-04-09 08:30:08,OPP-0002492,2024-03-03 09:43:18
1,ACT-00000014,2023-11-26 03:50:33,OPP-0002064,2025-02-02 14:54:02
2,ACT-00000015,2024-12-03 07:26:03,OPP-0002647,2025-09-17 22:48:35
3,ACT-00000017,2023-11-27 15:50:11,OPP-0001735,2024-12-25 21:06:07
4,ACT-00000019,2023-05-20 04:52:00,OPP-0002531,2024-05-26 22:29:58
...,...,...,...,...
3785,ACT-00019959,2023-02-01 08:23:48,OPP-0002455,2024-03-16 15:53:13
3786,ACT-00019962,2023-03-20 11:49:08,OPP-0000889,2025-08-17 14:36:46
3787,ACT-00019966,2024-10-13 09:59:31,OPP-0000423,2024-11-12 07:53:12
3788,ACT-00019989,2024-01-18 11:11:32,OPP-0000774,2024-06-21 00:54:47
